# Session 10 · Python for Data Mining
### Ecosystem, environments, importing & exporting data

**Data Mining 2026/27** · prof. dr. Uroš Godnov · Mon 14 Dec / Wed 16 Dec 2026 · 3 × 45 min + 15 min break

**From clicking to coding.** In AI Studio you connected operators into a process. From today you write the same steps as Python code — starting with the most underrated skill in data mining: getting data **into** Python from almost anywhere, and **out** again in a form other people can use.

| Part | Time | What we do |
|---|---|---|
| Warm-up | 5 min | from AI Studio operators to Python libraries |
| **A — Guided walkthrough** | 80 min | ecosystem · environments · Python refresher · importing from the web · first look · exporting |
| ☕ Break | 15 min | |
| **B — Your turn** | 35 min | Telco customer churn: a small import → clean → export pipeline |
| Wrap-up | 10 min | key takeaways, exit ticket, what's next |
| 🚀 **C — Advanced track (optional)** | after the wrap-up, or at home | robust data I/O: a download cache, file formats, DuckDB, polars, a data contract, official statistics from Eurostat |

**Learning objectives** — after this session you can:
1. work confidently in Colab and explain how it relates to a local Python installation;
2. map AI Studio operators to the Python data-mining ecosystem;
3. explain packages and environments, install a package and record versions for reproducibility;
4. import data from the web as CSV, Excel and JSON (API) and handle common gotchas: separators, decimal commas, encodings and wrongly typed columns;
5. export data to CSV, Excel (several sheets), Parquet and SQLite and read it back;
6. run a first-look routine on any DataFrame.

**How to read this notebook**

| Icon | Meaning |
|---|---|
| 🔮 | **Predict before you run** — make a guess, then run the cell and compare |
| 🧪 | **Try it** — a 2–5 minute exercise during the walkthrough |
| ✏️ | **TODO** — your task in Part B |
| ✅ | **Self-check** — run it for instant feedback (⏳ = not attempted yet) |
| 🛟 | **Rescue cell** — run only if you are stuck, so you can continue with the next task |
| 🤖 | **AI rule** — try on your own first; then Gemini may *explain* or give a *hint* (not write the solution) |
| 🚀 | **Fast lane** (optional) — side quests during Part A and the Part C challenges, for students who already code comfortably; never graded |
| 📈 💻 | **Two lenses** — a one-minute link to what you already know: 📈 economics and statistics, 💻 programming and databases |

> 💾 First thing: **File → Save a copy in Drive**, so your work is kept.

**Data used today** — all loaded from the web: Wine Quality (UCI) · Energy Efficiency (UCI, Excel) · Open-Meteo weather API · Telco Customer Churn (IBM) · in Part C also Eurostat inflation data (HICP).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/urosgodnov/dm-2627-python/blob/main/notebooks/s10-python-for-data-mining.ipynb)

### 🧭 Two lanes — pick yours

| Lane | For whom | How to work through this notebook |
|---|---|---|
| 🟢 **Core** | everyone — especially if Python is still new to you | Part A with the lecturer → Part B → wrap-up. This is everything the assignment and the project need. |
| 🚀 **Fast lane** | you already write Python comfortably (functions, loops, a bit of pandas) — e.g. you passed the 🚀 check in S00 | Follow Part A, and while we cover basics you know, work on the 🚀 **side quests**. Do Part B quickly (it is still the core of the project), then go on to **Part C · 🚀 Advanced track** after the wrap-up. |

You can switch lanes at any time. Everything marked 🚀 is optional, never graded, and its solutions are in the solutions notebook. Use the 📑 table of contents (left sidebar) to jump between parts.

**Two lenses.** 📈 boxes link the topic to economics and statistics, 💻 boxes to programming and databases. Read the one for your programme, or both. In Part C, 📈 challenges need more statistics and less code; 💻 challenges need more programming. **Strong in statistics but new to Python?** Stay in the 🟢 core for the code, and after the wrap-up try the 🧠 think questions and the 📈 challenges.

## 0 · Warm-up (5 min)

> 🔮 **Predict before you run:** In AI Studio, a process may contain **Read CSV → Normalize → Decision Tree**. Which Python libraries do you think do these three jobs? Write your guess in your notes or the chat.

## ⚙️ Setup
Run the next two cells first: the first imports today's libraries, the second loads the helper behind the ✅ self-checks.

In [ ]:
import io
import sqlite3
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 30)
print("Python", sys.version.split()[0], "| pandas", pd.__version__)

In [ ]:
#@title 🧰 Course helpers — run this cell (no need to read it) { display-mode: "form" }
CHECK_RESULTS = {}  # label -> True (correct) / False (not yet) / None (not attempted)

def check(label, test, hint=""):
    """Give instant feedback on an exercise without ever stopping 'Run all'.

    `test` is a function (usually a lambda) that returns True when the answer is right.
    """
    try:
        ok = bool(test())
    except NameError as err:
        CHECK_RESULTS[label] = None
        print(f"⏳ {label}: not attempted yet ({err}).")
        return
    except Exception as err:  # wrong type, wrong shape, missing column, ...
        CHECK_RESULTS[label] = False
        print(f"❌ {label}: checking your answer raised {type(err).__name__}: {err}")
        if hint:
            print(f"   💡 Hint: {hint}")
        return
    CHECK_RESULTS[label] = ok
    if ok:
        print(f"✅ {label}: correct!")
    else:
        print(f"❌ {label}: not yet.")
        if hint:
            print(f"   💡 Hint: {hint}")

def progress(*, return_count=False):
    """Summarise all self-checks run so far; optional 🚀 fast-lane checks are counted separately."""
    core = [v for label, v in CHECK_RESULTS.items() if not label.startswith("🚀")]
    fast = [v for label, v in CHECK_RESULTS.items() if label.startswith("🚀")]
    solved = sum(1 for v in core if v)
    print(f"Solved {solved} of {len(core)} exercises checked so far.")
    if fast:
        print(f"🚀 Fast lane (optional): solved {sum(1 for v in fast if v)} of {len(fast)}.")
    for label, v in CHECK_RESULTS.items():
        print(f"  {'✅' if v else ('⏳' if v is None else '❌')} {label}")
    if return_count:
        return solved

print("Helpers loaded ✔")

---
# Part A · Guided walkthrough

## A1 · Your runtime and the Python data-mining ecosystem (10 min)

**Google Colab** is a Jupyter notebook running on a machine in Google's cloud. Nothing is installed on your laptop: the machine — the **runtime** — comes with Python and hundreds of packages preinstalled. If you leave the tab idle for a while, the runtime is recycled. Your notebook (saved in Drive) stays, but **variables and files in the runtime are gone** — which is why every notebook starts with setup cells.

**From operators to code**

| AI Studio operator | Python | Library |
|---|---|---|
| Read CSV / Read Excel | `pd.read_csv()`, `pd.read_excel()` | pandas |
| Select Attributes · Filter Examples | `df[["a", "b"]]` · `df[df["a"] > 3]` | pandas |
| Replace Missing Values | `df.fillna()` · `SimpleImputer` | pandas · scikit-learn |
| Normalize | `StandardScaler` · `MinMaxScaler` | scikit-learn |
| Split Data | `train_test_split()` | scikit-learn |
| Decision Tree · Random Forest | `DecisionTreeClassifier` · `RandomForestClassifier` | scikit-learn |
| Gradient Boosted Trees | `XGBClassifier` | xgboost |
| Cross Validation | `cross_val_score()` | scikit-learn |
| Performance (Classification) | `classification_report()` · `roc_auc_score()` | scikit-learn |
| Auto Model | `TabularPredictor` | AutoGluon (S14) |

The core stack, bottom to top: **NumPy** (fast arrays) → **pandas** (tables) → **matplotlib / seaborn** (plots) → **scikit-learn** (preprocessing, models, evaluation) → **XGBoost / LightGBM** (boosting) → **AutoGluon** (AutoML).

> 🔮 **Predict before you run:** The next cell lists the versions of eight packages in this runtime. Which one do you expect to be **not installed**?

In [ ]:
for pkg in ["numpy", "pandas", "matplotlib", "seaborn", "scikit-learn", "xgboost", "lightgbm", "autogluon.tabular"]:
    try:
        print(f"{pkg:18s} {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:18s} — not installed")

> 📈 **Economist's view.** Python takes the place of Stata, EViews or R from your econometrics courses: `statsmodels` prints regression tables much like Stata's, and pandas reads the same Excel and CSV files. The difference is reach: the same free language also does machine learning, web data and automation.

> 💻 **Informatics view.** Compared with Java or C#, Python is dynamically typed and interpreted: there is no compile step, and in a notebook you run one cell at a time and inspect every result. Its packages come from PyPI (next section), the way Maven Central serves Java and NuGet serves C#.

## A2 · Installation and environments (15 min)

Three ideas you need — in Colab and on your own laptop:

1. **Interpreter** — the `python` program that runs your code (Colab currently runs Python 3.13).
2. **Packages** — add-on libraries such as pandas or scikit-learn, installed with **pip** from PyPI (the Python Package Index).
3. **Environment** — an isolated folder with *one* interpreter and *its own* set of packages. Every project gets its own environment, so upgrading pandas for project A cannot break project B.

In Colab the runtime **is** your environment, and it is fresh every session: whatever you `pip install` has to be installed again next time. That is why install lines live at the top of a notebook.

On your own laptop the modern, free setup is **uv** (installs Python, creates environments and installs packages — very fast) or **Miniforge** (conda), plus **VS Code** with the *Python* and *Jupyter* extensions, which opens the same `.ipynb` files you use in Colab.

👉 **Live demo (10 min):** your lecturer sets up a local environment with uv + VS Code. The step-by-step guide is in *Going further* at the end of this notebook.

In [ ]:
# Is a package already installed in this runtime? (missingno draws missing-value charts — we use it in S12)
%pip show missingno

In [ ]:
# Install a package that is NOT preinstalled: ucimlrepo downloads datasets from the UCI repository.
%pip install -q ucimlrepo
print("ucimlrepo", version("ucimlrepo"))

**Reproducibility** means that you — or a colleague, or your future self — get the same result when running the notebook again. Two habits make that possible:
- **record package versions** (in a project: `pip freeze > requirements.txt`, or `uv` does it for you);
- **fix random seeds** wherever randomness is involved (data splits, random forests, …).

In [ ]:
reqs = "\n".join(f"{p}=={version(p)}" for p in ["pandas", "numpy", "scikit-learn", "xgboost"])
Path("requirements.txt").write_text(reqs + "\n")
print(Path("requirements.txt").read_text())

> 🔮 **Predict before you run:** The next cell throws five dice with a **seeded** random number generator (`seed = 42`). If you run the cell twice, do you get the same five numbers?

In [ ]:
rng = np.random.default_rng(42)      # a seeded random number generator
print("Five dice throws:", rng.integers(1, 7, size=5), "← the same every time you rerun this cell")

## A3 · Python refresher — with data (10 min)

Four building blocks appear in every data-mining notebook: **lists**, **dicts**, **comprehensions** and **functions**. A list of dicts is also exactly what most web APIs return as JSON.

In [ ]:
# A tiny dataset: one dict per hotel booking
bookings = [
    {"guest": "Ana",   "nights": 3, "price_per_night": 92.0,  "canceled": False},
    {"guest": "Marko", "nights": 1, "price_per_night": 120.0, "canceled": True},
    {"guest": "Lea",   "nights": 7, "price_per_night": 75.5,  "canceled": False},
    {"guest": "Tim",   "nights": 2, "price_per_night": 99.0,  "canceled": False},
]

def revenue(booking):
    # Revenue of one booking: 0 if it was canceled.
    return 0.0 if booking["canceled"] else booking["nights"] * booking["price_per_night"]

revenues = [revenue(b) for b in bookings]                          # list comprehension
print(revenues, "→ total", sum(revenues))

long_stays = [b["guest"] for b in bookings if b["nights"] >= 3]     # comprehension with a filter
print("Long stays:", long_stays)

The same data as a pandas **DataFrame** — pandas' table with named columns and one row per record, like an *ExampleSet* in AI Studio. Instead of looping over the rows, we compute a whole column at once (this is called **vectorised** code).

In [ ]:
# The same table as a pandas DataFrame: vectorised columns instead of loops
df_bookings = pd.DataFrame(bookings)
df_bookings["revenue"] = np.where(df_bookings["canceled"], 0.0,
                                  df_bookings["nights"] * df_bookings["price_per_night"])
df_bookings

> ⚠️ **Common mistake:** using `print()` instead of `return` inside a function. `print` only *shows* a value on the screen; the function still gives back `None`, so a list built from it becomes `[None, None, None, None]`.

In [ ]:
# 🧪 Try it: Write a function `nights_category(n)` that returns "short" for 1–2 nights,
#    "medium" for 3–5 nights and "long" for 6 or more. Then build a list `categories`
#    with the category of every booking (in the same order as `bookings`).
# def nights_category(n):
#     ...
# categories = [...]


In [ ]:
#@title ✅ Check — A3 Try it { display-mode: "form" }
check('A3 Try it', lambda: categories == ["medium", "short", "long", "short"] and nights_category(6) == "long", hint='Use if / elif / else (or early returns) inside the function, then a list comprehension over bookings.')

In [ ]:
# 🚀 Side quest: pandas' groupby in pure Python. Write `total_by(rows, key, value)` that returns a dict
#    {group: sum of value(row)} — `key` and `value` are FUNCTIONS that take one booking.
#    Use `collections.defaultdict(float)`. Then compute
#    (a) `revenue_by_length`: revenue per stay length ("long" = 3 nights or more, otherwise "short");
#    (b) `stay_counts`: a `collections.Counter` with the number of bookings per stay length.
#    Compare (a) with a pandas groupby on `df_bookings`.
# from collections import Counter, defaultdict
#
# def total_by(rows, key, value):
#     totals = defaultdict(float)
#     ...
#     return dict(totals)
#
# revenue_by_length = total_by(bookings, lambda b: "long" if b["nights"] >= 3 else "short", revenue)
# stay_counts = Counter(...)


In [ ]:
#@title ✅ Check — 🚀 A3 side quest { display-mode: "form" }
check('🚀 A3 side quest', lambda: total_by(bookings, lambda b: b["canceled"], lambda b: b["nights"]) == {False: 12, True: 1} and revenue_by_length == {"long": 804.5, "short": 198.0} and stay_counts == {"long": 2, "short": 2}, hint='Inside the loop: totals[key(row)] += value(row). Counter() counts the items of any iterable.')

## A4 · Importing data from the web (30 min)

Most data arrives as a file behind a URL or as JSON from an **API** (a web address that programs — not people — ask for data). pandas reads directly from URLs: there is no separate download step.

### A4.1 CSV files — and the separator trap

The **Wine Quality** data (Cortez et al., 2009) contains 1,599 Portuguese red wines: 11 laboratory measurements and a quality score given by tasters.

In [ ]:
WINE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

> 🔮 **Predict before you run:** We read the file with pandas' default settings. How many columns will the DataFrame have?

In [ ]:
wine_wrong = pd.read_csv(WINE_URL)
print(wine_wrong.shape)
wine_wrong.head(2)

When something looks off, **look at the raw text** before guessing:

In [ ]:
raw = requests.get(WINE_URL, timeout=30).text
print(raw[:250])

The first line holds the column names (in quotes), and **semicolons** separate the values. So we tell pandas: `sep=";"`.

In [ ]:
wine = pd.read_csv(WINE_URL, sep=";")
print(wine.shape)
wine.head()

In [ ]:
# 🚀 Side quest: Real files arrive with ",", ";", tabs or "|". Write `read_csv_auto(text)` that detects the
#    separator with `csv.Sniffer().sniff(text[:2000], delimiters=",;\t|")` and returns
#    `pd.read_csv(io.StringIO(text), sep=...)`. Test it on `raw` (the wine text is already downloaded):
#    store the result as `wine_sniffed`. Then try it on a small tab-separated string of your own.
#    Bonus: `pd.read_csv(WINE_URL, sep=None, engine="python")` sniffs inside pandas. What does it cost?
# import csv
#
# def read_csv_auto(text):
#     dialect = csv.Sniffer().sniff(text[:2000], delimiters=",;\t|")
#     ...
#
# wine_sniffed = read_csv_auto(raw)


In [ ]:
#@title ✅ Check — 🚀 A4.1 side quest { display-mode: "form" }
check('🚀 A4.1 side quest', lambda: wine_sniffed.shape == (1599, 12) and read_csv_auto("a|b\n1|2\n").shape == (1, 2) and read_csv_auto("a;b;c\n1;2;3\n4;5;6\n").shape == (2, 3), hint='dialect.delimiter holds the detected separator; wrap the text in io.StringIO() for read_csv.')

### A4.2 "European" CSV files: decimal commas and character encodings

Slovenian (and most European) Excel exports use `;` as the separator and `,` as the decimal mark. Older systems also save text as **cp1250** (Windows Central European) instead of **UTF-8**, today's standard. An *encoding* is the rule that turns letters into bytes, so reading with the wrong rule garbles the letters. Let's simulate such a file (illustrative numbers):

In [ ]:
european_bytes = (
    "Občina;Prihodi turistov;Povprečna doba (dni)\n"
    "Koper;125.300;2,9\n"
    "Piran;512.840;3,4\n"
    "Škofja Loka;18.750;1,8\n"
).encode("cp1250")

`.dtypes` lists the type of each column: `int64` = whole numbers, `float64` = decimal numbers, `object` (pandas 2) or `str` (pandas 3) = text.

> 🔮 **Predict before you run:** What goes wrong if we read these bytes as UTF-8 with the default number settings? Look at the letters č/š and at the numbers.

In [ ]:
wrong = pd.read_csv(io.BytesIO(european_bytes), sep=";", encoding="utf-8", encoding_errors="replace")
print(wrong.dtypes)
wrong

In [ ]:
right = pd.read_csv(io.BytesIO(european_bytes), sep=";", encoding="cp1250", decimal=",", thousands=".")
print(right.dtypes)
right

> ⚠️ **Common mistake:** trusting a file because it loaded *without an error*. Above, `125.300` silently became 125.3 — no warning at all. After every import, look at `.dtypes` and at a few rows (`.head()`) before you continue.

In [ ]:
# 🚀 Side quest: Files rarely say which encoding they use. Write `decode_bytes(data, encodings=("utf-8", "cp1250"))`
#    that tries each encoding in order and returns `(text, encoding)` for the first one that works
#    (catch `UnicodeDecodeError`). Use it to read `european_bytes` into a DataFrame `towns` (with correct numbers).
#    Then ask the `charset_normalizer` package (installed together with requests) for its guess:
#    `charset_normalizer.from_bytes(european_bytes).best().encoding`. Is it right? Why does the ORDER of your list matter?
# import charset_normalizer
#
# def decode_bytes(data, encodings=("utf-8", "cp1250")):
#     for enc in encodings:
#         try:
#             ...
#         except UnicodeDecodeError:
#             ...
#
# towns_text, towns_encoding = decode_bytes(european_bytes)
# towns = ...


In [ ]:
#@title ✅ Check — 🚀 A4.2 side quest { display-mode: "form" }
check('🚀 A4.2 side quest', lambda: decode_bytes(european_bytes)[1] == "cp1250" and decode_bytes("Škofja Loka".encode("utf-8"))[1] == "utf-8" and towns["Prihodi turistov"].sum() == 656890, hint='return data.decode(enc), enc inside try; on UnicodeDecodeError continue with the next encoding.')

### A4.3 Excel files

`pd.read_excel()` reads `.xlsx` files from disk or from a URL (using the `openpyxl` package). The UCI **Energy Efficiency** data (Tsanas & Xifara, 2012) describes 768 simulated buildings; the targets are heating and cooling load. The column names are anonymous, so we rename them with a dict.

In [ ]:
ENB_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00242/ENB2012_data.xlsx"
energy = pd.read_excel(ENB_URL)
energy.head(3)

In [ ]:
energy = energy.rename(columns={
    "X1": "relative_compactness", "X2": "surface_area", "X3": "wall_area", "X4": "roof_area",
    "X5": "overall_height", "X6": "orientation", "X7": "glazing_area",
    "X8": "glazing_area_distribution", "Y1": "heating_load", "Y2": "cooling_load",
})
energy.head(3)

💡 Workbooks with several sheets: `pd.read_excel(path, sheet_name=None)` returns a dict `{sheet name: DataFrame}`.

### A4.4 JSON from a web API

APIs answer with **JSON**: nested dicts and lists, exactly the structures from A3. The free **Open-Meteo** API (no key needed) serves historical weather for any coordinates. Let's get the daily weather in **Koper** for 2025.

In [ ]:
METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 45.55, "longitude": 13.73,                       # Koper
    "start_date": "2025-01-01", "end_date": "2025-12-31",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "Europe/Ljubljana",
}
response = requests.get(METEO_URL, params=params, timeout=30)
print(response.status_code)            # 200 = OK
data = response.json()                 # JSON text -> Python dict
print(list(data.keys()))
print(data["daily_units"])
print("Grid cell:", data["latitude"], data["longitude"], "| elevation:", data["elevation"], "m")

In [ ]:
weather = pd.DataFrame(data["daily"])
weather["date"] = pd.to_datetime(weather["time"])
weather = weather.drop(columns="time")
weather.head()

In [ ]:
weather.plot(x="date", y=["temperature_2m_max", "temperature_2m_min"], figsize=(9, 3.5),
             title="Koper, 2025: daily maximum and minimum temperature",
             xlabel="date", ylabel="temperature (°C)")
plt.show()

**So what?** A few lines of code gave us a full year of daily weather: hot summer days of up to about 37 °C, winter nights around 0 °C. Note the *Grid cell* line above: the API answers for the nearest point of a weather model's grid (about 45.52 N, 13.70 E), so these are **modelled** values, not readings from a weather station in Koper. Always ask where data come from.

*Weather data by [Open-Meteo.com](https://open-meteo.com/) (CC BY 4.0).*

> 📈 **Economist's view.** Official statistics arrive the same way: Eurostat, the ECB and SURS publish their tables through web APIs and as CSV downloads, often in the European format from A4.2. A notebook that reads them straight from the URL updates itself when a new month is published (try C6 in Part C).

> 💻 **Informatics view.** Open-Meteo is a typical REST API: an HTTP GET request with query parameters returns JSON, just like the back end of a web app. `requests` is Python's HTTP client (like `HttpClient` in C# or Java), and status code 200 is the HTTP status you know from web development.

In [ ]:
# 🧪 Try it: Request the same data for Ljubljana (latitude 46.05, longitude 14.51) and store the
#    DataFrame as `weather_lj`. Which city had more precipitation in 2025?
#    Tip: `{**params, "latitude": ...}` copies a dict and overrides some keys.
# params_lj = ...
# weather_lj = ...


In [ ]:
#@title ✅ Check — A4 Try it { display-mode: "form" }
check('A4 Try it', lambda: len(weather_lj) == 365 and "precipitation_sum" in weather_lj.columns, hint='Build params_lj, call requests.get(...).json(), and turn the "daily" part into a DataFrame.')

## A5 · The first-look routine (8 min)

Whenever you load a new dataset, run the same handful of commands **before anything else**. They answer: *How big? Which types? Anything missing? Anything odd?* It is the code version of the *Statistics* view of a result in AI Studio.

In [ ]:
print(wine.shape)
wine.info()

In [ ]:
wine.describe().T

Read `describe()` column by column. For example, `total sulfur dioxide` has a median (`50%`) of 38 but a maximum of 289: a long tail of unusually high values. Such columns deserve a plot before modelling (S11).

> 🔮 **Predict before you run:** The wine data has no missing values. Do you expect **duplicate rows**? Why could a lab dataset contain identical rows?

In [ ]:
print("Missing values per column:", int(wine.isna().sum().sum()))
print("Duplicate rows:", wine.duplicated().sum())
wine["quality"].value_counts().sort_index()

**So what?** The target is **imbalanced**: 1,319 of the 1,599 wines (82 %) have quality 5 or 6, while only 10 have quality 3 and 18 have quality 8. A model will rarely see the extremes, so it will be weakest exactly there.

In [ ]:
# 🧪 Try it: Run the first-look routine on `energy`. How many distinct values does `orientation` take?
#    Store the answer in `n_orientations`.
# n_orientations = ...


In [ ]:
#@title ✅ Check — A5 Try it { display-mode: "form" }
check('A5 Try it', lambda: n_orientations == 4, hint='Use .nunique() on the orientation column.')

## A6 · Exporting data: CSV, Excel, Parquet, SQL (7 min)

Results must leave Python, too (in AI Studio: *Write CSV*, *Write Excel*, *Write Database*): for colleagues (**Excel**), for other programs (**CSV**), for large data and fast reloading (**Parquet**) and for databases (**SQL**).

> 🔮 **Predict before you run:** We save the same wine table as CSV, Parquet and Excel. Which file will be the **smallest**?

In [ ]:
wine.to_csv("wine.csv", index=False)
wine.to_parquet("wine.parquet", index=False)

with pd.ExcelWriter("wine_report.xlsx") as writer:          # one file, two sheets
    wine.to_excel(writer, sheet_name="data", index=False)
    wine.describe().T.to_excel(writer, sheet_name="summary")

for f in ["wine.csv", "wine.parquet", "wine_report.xlsx"]:
    print(f"{f:18s} {Path(f).stat().st_size / 1024:7.1f} KB")

**Parquet** is a compressed, column-by-column binary format that also stores the column types: here it needs about a third of the space of the CSV (32 KB vs 91 KB). Excel is meant for people, not for programs: it is the largest and the slowest to read.

> ⚠️ **Common mistake:** forgetting `index=False`. pandas then also writes the row numbers 0, 1, 2, … as an extra first column, which comes back as `Unnamed: 0` when you read the file again.

In [ ]:
back = pd.read_parquet("wine.parquet")
print("Identical after the round trip:", back.equals(wine))

In [ ]:
con = sqlite3.connect(":memory:")           # a throwaway SQL database in memory
wine.to_sql("wine", con, index=False)
pd.read_sql('''
    SELECT quality, COUNT(*) AS n, ROUND(AVG(alcohol), 2) AS avg_alcohol
    FROM wine
    GROUP BY quality
    ORDER BY quality
''', con)

**So what?** Better wines contain more alcohol on average: 9.9 % for quality 5 but 12.1 % for quality 8 — a first hint of a useful feature for a model.

> 📈 **Economist's view.** pandas also speaks Stata: `pd.read_stata()` and `df.to_stata("data.dta")` move a cleaned dataset between Python and Stata in one line. For a thesis or a paper, the notebook plus a Parquet or CSV file is a complete replication package.

> 💻 **Informatics view.** These formats map onto a data warehouse: SQLite is a single-file relational database with largely the same SQL as PostgreSQL or SQL Server, and Parquet is the standard file format of data lakes, which BigQuery, Snowflake or Databricks query directly. `to_sql` also writes to a real database server once you pass it a SQLAlchemy engine instead of a `sqlite3` connection.

In [ ]:
# 🚀 Side quest: Never paste user input into SQL with an f-string. A web form asks for a minimum quality.
#    (a) Write `count_wines(min_quality)` that returns (as an int) the number of wines with quality >= min_quality,
#        using a placeholder: `pd.read_sql("... WHERE quality >= ?", con, params=(min_quality,))`.
#    (b) A user types `7 OR 1=1`. Compare `count_wines("7 OR 1=1")` with the f-string version
#        `pd.read_sql(f"SELECT COUNT(*) AS n FROM wine WHERE quality >= {user_input}", con)`. What happened?
# def count_wines(min_quality):
#     result = pd.read_sql("SELECT COUNT(*) AS n FROM wine WHERE quality >= ?", con, params=(min_quality,))
#     return ...
#
# user_input = "7 OR 1=1"
# ...


In [ ]:
#@title ✅ Check — 🚀 A6 side quest { display-mode: "form" }
check('🚀 A6 side quest', lambda: count_wines(7) == 217 and count_wines(8) == 18 and count_wines("7 OR 1=1") == 0, hint='params=(min_quality,) needs the comma: a tuple with one element. Return int(result.loc[0, "n"]).')

In Colab, download a file with `from google.colab import files; files.download("wine_report.xlsx")` or via the 📁 folder icon on the left. Files in the runtime disappear when it is recycled — keep anything important in Google Drive (see *Going further*).

> **Before the break:** read the Part B scenario below, so you can start right away.

---
## ☕ Break — 15 minutes

---
# Part B · Your turn: Telco customer churn (35 min)

**Scenario.** You are a junior analyst at a telecom operator. The CRM team sends you a link to a customer export (IBM's public *Telco Customer Churn* sample, 7,043 customers). Before anyone builds a churn model, the team needs a clean, documented copy of the data in formats the rest of the company can open.

Work through Tasks 1–5 and run the ✅ check after each one. Stuck for more than 3 minutes? Ask your neighbour, then Gemini (*explain* or *hint*, not *solve*), or run the 🛟 rescue cell and continue with the next task.

In [ ]:
TELCO_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

In [ ]:
# ✏️ TODO: Task 1 — Load the Telco data from TELCO_URL into a DataFrame called `telco`,
#    then print its shape and its column types (`.dtypes`).
# telco = ...


In [ ]:
#@title ✅ Check — Task 1 { display-mode: "form" }
check('Task 1', lambda: telco.shape == (7043, 21), hint='pd.read_csv(TELCO_URL) — this file uses commas, so no sep= is needed.')

**✍️ Your answer** — Which column has a surprising data type? (Look for a number stored as text.)

> *(double-click this cell and write your answer here)*

In [ ]:
#@title 🛟 Rescue — run only if you could not finish Task 1 { display-mode: "form" }
telco = pd.read_csv(TELCO_URL)
print("telco loaded:", telco.shape)

In [ ]:
# ✏️ TODO: Task 2 — Make a copy `telco_clean` in which `TotalCharges` is converted to numbers with
#    pd.to_numeric(..., errors="coerce"). Store the number of missing values this creates in `n_missing_total`.
#    Then display those rows together with their `tenure`.
# telco_clean = telco.copy()
# telco_clean["TotalCharges"] = ...
# n_missing_total = ...


In [ ]:
#@title ✅ Check — Task 2 { display-mode: "form" }
check('Task 2', lambda: n_missing_total == 11 and pd.api.types.is_numeric_dtype(telco_clean["TotalCharges"]), hint='errors="coerce" turns text that is not a number (here: blanks) into NaN; count them with .isna().sum().')

**✍️ Your answer** — All of these customers share the same `tenure`. Why is `TotalCharges` blank, and which value would be sensible?

> *(double-click this cell and write your answer here)*

In [ ]:
#@title 🛟 Rescue — run only if you could not finish Task 2 { display-mode: "form" }
telco_clean = telco.copy()
telco_clean["TotalCharges"] = pd.to_numeric(telco_clean["TotalCharges"], errors="coerce")
print("telco_clean ready:", telco_clean["TotalCharges"].dtype)

In [ ]:
# ✏️ TODO: Task 3 — Write a reusable function `quick_profile(df)` that returns a DataFrame with one row per
#    column of `df` and the columns `dtype`, `n_missing`, `pct_missing` (0–100, rounded to 1 decimal)
#    and `n_unique`. Test it on `telco_clean`. You will reuse this function in S11!
# def quick_profile(df):
#     return pd.DataFrame({
#         "dtype": ...,
#         "n_missing": ...,
#         "pct_missing": ...,
#         "n_unique": ...,
#     })


In [ ]:
#@title ✅ Check — Task 3 { display-mode: "form" }
check('Task 3', lambda: list(quick_profile(telco_clean).columns) == ["dtype", "n_missing", "pct_missing", "n_unique"] and quick_profile(telco_clean).loc["TotalCharges", "n_missing"] == 11, hint='Build the DataFrame from a dict of Series: df.dtypes, df.isna().sum(), df.isna().mean(), df.nunique().')

In [ ]:
#@title 🛟 Rescue — run this if you restarted the runtime (restores the Open-Meteo settings from Part A) { display-mode: "form" }
METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 45.55, "longitude": 13.73,
    "start_date": "2025-01-01", "end_date": "2025-12-31",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "Europe/Ljubljana",
}
print("API settings from Part A restored.")

In [ ]:
# ✏️ TODO: Task 4 — Fetch the 2025 daily weather for a city of your choice from Open-Meteo (find the
#    coordinates on latlong.net or ask Gemini). Store it as `weather_city` with a datetime column `date`,
#    then compute the monthly mean of the daily maximum temperature as `monthly_max` (12 rows).
# params_city = {**params, "latitude": ..., "longitude": ...}
# weather_city = ...
# monthly_max = ...


In [ ]:
#@title ✅ Check — Task 4 { display-mode: "form" }
check('Task 4', lambda: len(monthly_max) == 12 and "date" in weather_city.columns, hint='Group by weather_city["date"].dt.month and take the mean of temperature_2m_max.')

In [ ]:
#@title 🛟 Rescue — run only if Task 2 or Task 3 is not solved (restores quick_profile and telco_clean) { display-mode: "form" }
def quick_profile(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(),
    })

if "telco_clean" not in globals():
    telco_clean = pd.read_csv(TELCO_URL)
    telco_clean["TotalCharges"] = pd.to_numeric(telco_clean["TotalCharges"], errors="coerce")
print("quick_profile and telco_clean ready.")

In [ ]:
# ✏️ TODO: Task 5 — Export `telco_clean` to Parquet (`telco_clean.parquet`) and to an Excel file
#    `telco_report.xlsx` with two sheets: `data` (the cleaned data) and `profile` (the output of
#    quick_profile). Read the Parquet file back into `telco_back` and confirm it equals `telco_clean`.
# telco_clean.to_parquet(...)
# with pd.ExcelWriter("telco_report.xlsx") as writer:
#     ...
# telco_back = ...


In [ ]:
#@title ✅ Check — Task 5 { display-mode: "form" }
check('Task 5', lambda: telco_back.equals(telco_clean) and pd.ExcelFile("telco_report.xlsx").sheet_names == ["data", "profile"], hint='Use one pd.ExcelWriter block with two .to_excel(..., sheet_name=...) calls.')

In [ ]:
# ✏️ ⭐ Stretch: Load `telco_clean` into an in-memory SQLite database and write ONE SQL query that
#    returns the churn rate (%) per `Contract` type, highest first. Store the result as `churn_by_contract`.
# con = sqlite3.connect(":memory:")
# ...


In [ ]:
#@title ✅ Check — Stretch { display-mode: "form" }
check('Stretch', lambda: churn_by_contract.iloc[0]["Contract"] == "Month-to-month", hint="SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) counts churners; divide by COUNT(*).")

**✍️ Your answer** — What does the churn rate per contract type suggest to the business?

> *(double-click this cell and write your answer here)*

> 🚀 **Done early?** Join the wrap-up with everyone first, then continue with **Part C · 🚀 Advanced track** right after it: six challenges (plus an open one) that turn today's import and export code into robust tools.

---
## Wrap-up (10 min)

**Key takeaways**
1. Colab gives you a fresh Python environment in the cloud: install lines go at the top, and runtime files vanish unless you save them.
2. Environments isolate a project's packages. Record versions (`requirements.txt`) and fix seeds for reproducibility.
3. `read_csv` / `read_excel` / `read_parquet` / `read_sql` plus `requests` for APIs cover most data sources. Always check separators, decimal marks, encodings and dtypes.
4. Run the first-look routine before any analysis.
5. Export for people (Excel), programs (CSV), performance (Parquet) and databases (SQL).

In [ ]:
progress()

### 📋 Cheat sheet — today's essentials

| I want to … | Code |
|---|---|
| check a package's version | `version("pandas")` |
| install a package (in Colab: again in every new runtime) | `%pip install -q ucimlrepo` |
| read a CSV that uses semicolons | `pd.read_csv(url, sep=";")` |
| read a European CSV | `pd.read_csv(f, sep=";", decimal=",", thousands=".", encoding="cp1250")` |
| read every sheet of an Excel file | `pd.read_excel(url, sheet_name=None)` |
| get JSON from an API | `requests.get(url, params=params, timeout=30).json()` |
| turn text into numbers (bad values → NaN) | `pd.to_numeric(s, errors="coerce")` |
| first look at a new table | `df.shape; df.info(); df.describe().T; df.isna().sum(); df.duplicated().sum()` |
| save a CSV without the row numbers | `df.to_csv("f.csv", index=False)` |
| one Excel file with several sheets | `with pd.ExcelWriter("f.xlsx") as w: df.to_excel(w, sheet_name="data")` |
| save fast and keep the column types | `df.to_parquet("f.parquet", index=False); pd.read_parquet("f.parquet")` |
| query a table with SQL | `df.to_sql("t", con, index=False); pd.read_sql("SELECT ...", con)` |

> 🧩 **For your project:** start your project notebook with one *load* cell (URL → `read_csv` or `read_excel` with the right `sep`, `decimal` and `encoding`), then run the first-look routine and `quick_profile` before any modelling. Save the cleaned data as **Parquet**, so every team member starts from the same file with the same column types, and write the package versions to `requirements.txt`.

**✍️ Your answer** — **Exit ticket.** (1) Which argument fixes a file that loads as a single column? (2) Why fill the blank `TotalCharges` with 0 rather than the column mean? (3) Name one advantage of Parquet over CSV.

> *(double-click this cell and write your answer here)*

**Next time — S11 (Mon 21 Dec, on Zoom):** pandas & exploratory data analysis on 119,000 hotel bookings, plus the statistics to judge which features matter. Before the session, check that Colab and Zoom both work on the same laptop — and keep your `quick_profile` function.

## 🧠 Think it through (optional · no code · 10 min)

Questions for thinking rather than coding, e.g. if you are strong in statistics but new to Python. Discuss them in pairs, or write two or three sentences for each. The solutions notebook has model answers.

1. 📈 The 11 blank `TotalCharges` belong to brand-new customers. Suppose instead that most blanks belonged to customers who later left. Why would simply dropping those rows bias the churn rate you report?
2. 📈 Eurostat publishes inflation every month and can correct past values. Why does it matter for a policy report whether your notebook downloads the data at run time or reads a saved copy, and what should the report state?
3. 💻 Your notebook runs every morning on a server and updates a churn dashboard. Name two things from today that could break it overnight, and how you would protect the job.
4. 💻 `TotalCharges` arrives as text because 11 values are blank. Is that a problem of your notebook or of the system that exports the data, and where should it be fixed?

> ✍️ *(double-click this cell and write your answers here)*

---
# Part C · 🚀 Advanced track (optional)

**For:** fast-lane students who finished Part B — or anyone who wants more, at home. Never graded; the assignment and the project do not need it.

**Theme:** From scripts to tools: robust data I/O. Part A showed *how* to get data in and out. Here you build the small tools that make it reliable and fast: a download cache, an informed choice of file format, SQL without a database server, lazy queries, and a data contract that stops bad data before it reaches a model. And you read official statistics straight from Eurostat's API.

| # | Challenge | Time | Level | Lens |
|---|---|---|---|---|
| **C1** | A cached downloader: never download the same file twice | 15 min | 🌶️ | 💻 |
| **C2** | Which file format? CSV vs Parquet vs Feather vs pickle on 140k rows | 15 min | 🌶️ | 💻 |
| **C3** | SQL without a server: DuckDB on a DataFrame and on a Parquet file | 10 min | 🌶️ | 💻 |
| **C4** | polars, lazily: a query plan that reads only what it needs | 15 min | 🌶️🌶️ | 💻 |
| **C5** | A data contract: `validate(df, schema)` catches bad data | 20 min | 🌶️🌶️ | 💻 |
| **C6** | Official statistics from Eurostat: inflation in Slovenia vs the euro area vs the EU | 15 min | 🌶️ | 📈 |
| **🏁** | Open: several cities in one Open-Meteo request → tidy table + plot | 20 min | open | 📈💻 |

**Lens:** 📈 more statistics and economics, less code · 💻 more programming and data engineering · 📈💻 both

Each challenge stands on its own. **Run the ⚙️ Part C setup cell first**, then pick any challenge. The usual rules apply: a first own attempt, then Gemini may *explain* or give a *hint*.

DuckDB and polars are preinstalled in Colab: no `pip install` needed.

In [ ]:
#@title ⚙️ Part C setup — run this first (rebuilds everything Part C needs) { display-mode: "form" }
if "check" not in globals():  # runtime restarted? then restore the ✅ self-check helpers
    CHECK_RESULTS = {}  # label -> True (correct) / False (not yet) / None (not attempted)

    def check(label, test, hint=""):
        """Give instant feedback on an exercise without ever stopping 'Run all'.

        `test` is a function (usually a lambda) that returns True when the answer is right.
        """
        try:
            ok = bool(test())
        except NameError as err:
            CHECK_RESULTS[label] = None
            print(f"⏳ {label}: not attempted yet ({err}).")
            return
        except Exception as err:  # wrong type, wrong shape, missing column, ...
            CHECK_RESULTS[label] = False
            print(f"❌ {label}: checking your answer raised {type(err).__name__}: {err}")
            if hint:
                print(f"   💡 Hint: {hint}")
            return
        CHECK_RESULTS[label] = ok
        if ok:
            print(f"✅ {label}: correct!")
        else:
            print(f"❌ {label}: not yet.")
            if hint:
                print(f"   💡 Hint: {hint}")

    def progress(*, return_count=False):
        """Summarise all self-checks run so far; optional 🚀 fast-lane checks are counted separately."""
        core = [v for label, v in CHECK_RESULTS.items() if not label.startswith("🚀")]
        fast = [v for label, v in CHECK_RESULTS.items() if label.startswith("🚀")]
        solved = sum(1 for v in core if v)
        print(f"Solved {solved} of {len(core)} exercises checked so far.")
        if fast:
            print(f"🚀 Fast lane (optional): solved {sum(1 for v in fast if v)} of {len(fast)}.")
        for label, v in CHECK_RESULTS.items():
            print(f"  {'✅' if v else ('⏳' if v is None else '❌')} {label}")
        if return_count:
            return solved

    print("Helpers loaded ✔")

# Everything Part C needs: safe to run again at any time.
import hashlib
import re
import time
from pathlib import Path
from urllib.parse import urlparse

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import requests

WINE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
TELCO_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
METEO_URL = "https://archive-api.open-meteo.com/v1/archive"

# The cleaned Telco table from Part B (Task 2): rebuilt if it is missing or not cleaned yet
if not ("telco_clean" in globals() and isinstance(telco_clean, pd.DataFrame) and telco_clean.shape == (7043, 21)
        and pd.api.types.is_numeric_dtype(telco_clean["TotalCharges"])):
    telco_clean = pd.read_csv(TELCO_URL)
    telco_clean["TotalCharges"] = pd.to_numeric(telco_clean["TotalCharges"], errors="coerce")
telco_clean.to_parquet("telco_clean.parquet", index=False)  # the Parquet file from Task 5 (used in C3 and C4)

print("Part C ready ✔ | telco_clean", telco_clean.shape, "| duckdb", duckdb.__version__, "| polars", pl.__version__)

## C1 · A cached downloader 🌶️ (15 min) · 💻

**Why it matters.** Every *Run all* downloads the same files again: slow for you, impolite to the server, and impossible offline. Real projects keep a local **cache**: download once, then reuse the file.

**Build** `fetch(url, folder="data") -> Path` that
1. names the file after a **hash of the URL**: the first 12 characters of `hashlib.sha256(url.encode()).hexdigest()`, then `_` and the original file name, e.g. `data/b947035ba2ad_winequality-red.csv` (so two URLs that both end in `data.csv` cannot overwrite each other);
2. returns the cached file **without any network call** if it already exists;
3. otherwise downloads with a `timeout`, calls `response.raise_for_status()` **before** writing anything (an error page must never end up in your cache), saves the bytes and returns the path.

**Hints:** `Path(folder).mkdir(exist_ok=True)` · `Path(urlparse(url).path).name` is the file-name part of a URL · `path.write_bytes(response.content)`. Bonus: write to a temporary `.part` file first and rename it at the end (`tmp.replace(path)`), so an interrupted download never leaves half a file behind.

In [ ]:
# 🚀 Challenge: Write fetch(url, folder="data") -> Path with a hash-based file name, a timeout,
#    raise_for_status() and no second download. Then load the wine data through it.
# def fetch(url, folder="data", timeout=30):
#     folder = Path(folder)
#     folder.mkdir(exist_ok=True)
#     digest = hashlib.sha256(url.encode()).hexdigest()[:12]
#     path = folder / f"{digest}_{...}"
#     if path.exists():
#         return path
#     ...
#
# wine_path = fetch(WINE_URL)


In [ ]:
#@title ✅ Check — 🚀 C1 { display-mode: "form" }
def _test():
    first = Path(fetch(WINE_URL))
    real_request = requests.Session.request
    def no_network(*args, **kwargs):
        raise RuntimeError("fetch() went to the network although the file was already cached")
    requests.Session.request = no_network            # every requests call fails during the second fetch
    try:
        second = Path(fetch(WINE_URL))
    finally:
        requests.Session.request = real_request
    bad_url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/no-such-file.csv"
    try:
        fetch(bad_url)
        raised = False
    except requests.HTTPError:
        raised = True
    leftovers = list(Path("data").glob(hashlib.sha256(bad_url.encode()).hexdigest()[:12] + "*"))
    return (first == second and first.parent.name == "data"
            and first.name.startswith(hashlib.sha256(WINE_URL.encode()).hexdigest()[:12])
            and pd.read_csv(first, sep=";").shape == (1599, 12) and raised and not leftovers)

check('🚀 C1', _test, hint="Return the cached path before any request; call raise_for_status() before writing; name = sha256(url)[:12] + '_' + the file name.")

## C2 · Which file format? 🌶️ (15 min) · 💻

**Why it matters.** The format you save in decides how fast your notebook restarts, how much space your data takes, and whether the column types survive. Let's measure instead of guessing, on a table big enough to feel the difference (the next cell builds it).

**Build** a DataFrame `bench` with one row per format (index `csv`, `parquet`, `feather`, `pickle`) and the columns `write_s` and `read_s` (seconds), `size_mb` and `dtypes_kept` (True if every column comes back with the same dtype).

**Hints:** time with `t0 = time.perf_counter()` … `time.perf_counter() - t0` · `big.to_feather(p)` / `pd.read_feather(p)` · `big.to_pickle(p)` / `pd.read_pickle(p)` · size: `Path(p).stat().st_size / 1e6` · types: `back.dtypes.equals(big.dtypes)`. A dict `{name: (write_function, read_function)}` keeps the loop short.

In [ ]:
# A bigger table: Telco × 20 = 140,860 rows, with unique IDs and one 'category' column
big = pd.concat([telco_clean] * 20, ignore_index=True)
big["customerID"] = big["customerID"] + "-" + (big.index // len(telco_clean)).astype(str)
big["Contract"] = big["Contract"].astype("category")
print(big.shape, "|", round(big.memory_usage(deep=True).sum() / 1e6, 1), "MB in memory")

In [ ]:
# 🚀 Challenge: Benchmark CSV, Parquet, Feather and pickle on `big`: write time, read time, file size (MB) and
#    whether the dtypes survive the round trip. Store the results in `bench` (index = format name).
# formats = {
#     "csv": (lambda p: big.to_csv(p, index=False), pd.read_csv),
#     ...
# }
# rows = {}
# for name, (write, read) in formats.items():
#     ...
# bench = pd.DataFrame.from_dict(rows, orient="index")


In [ ]:
#@title ✅ Check — 🚀 C2 { display-mode: "form" }
check('🚀 C2', lambda: set(bench.index) == {"csv", "parquet", "feather", "pickle"} and not bench.loc["csv", "dtypes_kept"] and bench.loc[["parquet", "feather", "pickle"], "dtypes_kept"].all() and bench.loc["parquet", "size_mb"] < bench.loc["csv", "size_mb"] / 3, hint='One row per format with the columns write_s, read_s, size_mb, dtypes_kept; dtypes_kept = back.dtypes.equals(big.dtypes).')

**✍️ Your answer** — Which format would you choose (a) to send to a colleague, (b) for your project's cleaned data, (c) never for a file downloaded from the internet — and why?

> *(double-click this cell and write your answer here)*

## C3 · SQL without a server: DuckDB 🌶️ (10 min) · 💻

**Why it matters.** SQLite (A6) needed a copy of the data inside a database. **DuckDB** is an analytical SQL engine that runs *inside* Python and queries a pandas DataFrame — or a Parquet file on disk — directly, without loading it anywhere first.

**Build** the churn rate (%) per `Contract`, highest first, with the columns `Contract` and `churn_pct`:
1. `churn_duck_df` — from the DataFrame `telco_clean` (simply write its name after `FROM`);
2. `churn_duck_pq` — straight from the file: `FROM 'telco_clean.parquet'`;
3. compare both with a pandas `groupby`.

**Hints:** `duckdb.sql("SELECT ...").df()` returns a pandas DataFrame · `AVG(CASE WHEN Churn = 'Yes' THEN 100.0 ELSE 0 END)` · `ORDER BY churn_pct DESC`.

In [ ]:
# 🚀 Challenge: Compute the churn rate (%) per Contract with DuckDB: once from the DataFrame telco_clean
#    (churn_duck_df) and once from the file telco_clean.parquet (churn_duck_pq). Compare with pandas.
# churn_duck_df = duckdb.sql('''
#     SELECT Contract, ... AS churn_pct
#     FROM telco_clean
#     ...
# ''').df()
# churn_duck_pq = ...


In [ ]:
#@title ✅ Check — 🚀 C3 { display-mode: "form" }
def _test():
    reference = telco_clean["Churn"].eq("Yes").groupby(telco_clean["Contract"]).mean().mul(100)
    def rates(result):
        return dict(zip(result["Contract"], result["churn_pct"].astype(float)))
    same = all(abs(rates(result).get(contract, -1) - pct) < 1e-6
               for result in (churn_duck_df, churn_duck_pq) for contract, pct in reference.items())
    return same and list(churn_duck_df["Contract"]) == ["Month-to-month", "One year", "Two year"] and len(churn_duck_pq) == 3

check('🚀 C3', _test, hint="Columns Contract and churn_pct (in %), ORDER BY churn_pct DESC; for the file write FROM 'telco_clean.parquet'.")

## C4 · polars, lazily 🌶️🌶️ (15 min) · 💻

**Why it matters.** pandas runs every line immediately (*eager*). **polars** can also work *lazily*: you describe the whole query first, polars optimises the plan, and only `.collect()` runs it. One optimisation is *projection pushdown*: reading only the columns the query needs. On a 10 GB Parquet file with 50 columns, that is the difference between seconds and minutes.

**Build**
1. `churn_lazy` = `pl.scan_parquet("telco_clean.parquet")` → `.group_by("Contract")` → `.agg(...)` the churn rate in % named `churn_pct` → `.sort("churn_pct", descending=True)`. Nothing is read yet!
2. `plan_c4 = churn_lazy.explain()`: print it and find the line `PROJECT …/21 COLUMNS`;
3. `churn_polars = churn_lazy.collect()`: compare it with pandas.

**Hints:** `((pl.col("Churn") == "Yes").mean() * 100).alias("churn_pct")` · `print(plan_c4)`. Bonus (in a new variable): add `.filter(pl.col("InternetService") == "Fiber optic")` and see where the filter ends up in the plan.

In [ ]:
# 🚀 Challenge: Build the same churn-rate table with a LAZY polars query on telco_clean.parquet:
#    churn_lazy (a LazyFrame), plan_c4 = churn_lazy.explain(), churn_polars = churn_lazy.collect().
# churn_lazy = (
#     pl.scan_parquet("telco_clean.parquet")
#     .group_by("Contract")
#     .agg(...)
#     .sort(...)
# )
# plan_c4 = ...
# churn_polars = ...


In [ ]:
#@title ✅ Check — 🚀 C4 { display-mode: "form" }
def _test():
    import re
    reference = telco_clean["Churn"].eq("Yes").groupby(telco_clean["Contract"]).mean().mul(100)
    got = dict(zip(churn_polars["Contract"].to_list(), churn_polars["churn_pct"].to_list()))
    pushdown = re.search(r"PROJECT (\d+)/(\d+) COLUMNS", plan_c4)
    return (isinstance(churn_lazy, pl.LazyFrame) and pushdown is not None
            and int(pushdown.group(1)) < int(pushdown.group(2))
            and all(abs(got.get(contract, -1) - pct) < 1e-6 for contract, pct in reference.items())
            and churn_polars["Contract"].to_list()[0] == "Month-to-month")

check('🚀 C4', _test, hint='Keep the LazyFrame in churn_lazy (no .collect() there), plan_c4 = churn_lazy.explain(), churn_polars = churn_lazy.collect().')

**✍️ Your answer** — What does `PROJECT 2/21 COLUMNS` in the plan mean, and why does it matter for big data?

> *(double-click this cell and write your answer here)*

## C5 · A data contract 🌶️🌶️ (20 min) · 💻

**Why it matters.** Next month the CRM team sends a new export. Did a column disappear? Did someone type "Monthly" instead of "Month-to-month"? A model trains on broken data without complaint and gives wrong answers *silently*. A **data contract** is a written list of rules the data must satisfy, checked automatically every time new data arrives.

**Build** `validate(df, schema) -> list[str]`: one readable message per problem, **starting with the column name**, and an empty list when the data is fine. For every column in `schema`, check that
- the column exists (report a missing column, don't crash);
- `"dtype"` fits: `"number"` or `"text"`;
- values stay within `"min"` / `"max"` and inside the `"allowed"` set;
- values are not duplicated if `"unique"` is True;
- there are no missing values, unless `"nullable"` is True.

It must pass `telco_clean` (→ `[]`) and catch the three problems in `telco_broken` (next cell).

**Hints:** `pd.api.types.is_numeric_dtype(s)` and `pd.api.types.is_string_dtype(s)` · `(s < rules["min"]).sum()` · `set(s.dropna()) - set(rules["allowed"])` · `s.duplicated().sum()` · `s.isna().sum()` · start with `problems = []` and `problems.append(f"{col}: ...")`.

In [ ]:
TELCO_SCHEMA = {
    "customerID":     {"dtype": "text", "unique": True},
    "SeniorCitizen":  {"dtype": "number", "allowed": {0, 1}},
    "tenure":         {"dtype": "number", "min": 0, "max": 120},          # months
    "Contract":       {"dtype": "text", "allowed": {"Month-to-month", "One year", "Two year"}},
    "MonthlyCharges": {"dtype": "number", "min": 0, "max": 500},
    "TotalCharges":   {"dtype": "number", "min": 0, "nullable": True},    # 11 new customers have no bill yet
    "Churn":          {"dtype": "text", "allowed": {"Yes", "No"}},
}

telco_broken = telco_clean.copy()                    # "next month's export", with three problems
telco_broken.loc[3, "MonthlyCharges"] = -20.0
telco_broken.loc[10, "customerID"] = telco_broken.loc[0, "customerID"]
telco_broken.loc[25, "Contract"] = "Monthly"
print("Schema for", len(TELCO_SCHEMA), "columns; telco_broken is ready.")

In [ ]:
# 🚀 Challenge: Write validate(df, schema) -> list[str]: one message per problem (starting with the column name),
#    an empty list for good data. Test it on telco_clean and on telco_broken.
# def validate(df, schema):
#     problems = []
#     for col, rules in schema.items():
#         if col not in df.columns:
#             problems.append(f"{col}: column is missing")
#             continue
#         s = df[col]
#         ...
#     return problems
#
# print(validate(telco_clean, TELCO_SCHEMA))
# print(validate(telco_broken, TELCO_SCHEMA))


In [ ]:
#@title ✅ Check — 🚀 C5 { display-mode: "form" }
def _test():
    broken = validate(telco_broken, TELCO_SCHEMA)
    no_churn = validate(telco_clean.drop(columns="Churn"), TELCO_SCHEMA)               # a missing column
    text_tenure = validate(telco_clean.assign(tenure=telco_clean["tenure"].astype(str)), TELCO_SCHEMA)
    return (validate(telco_clean, TELCO_SCHEMA) == [] and len(broken) >= 3
            and all(any(col in msg for msg in broken) for col in ["MonthlyCharges", "customerID", "Contract"])
            and any("Churn" in msg for msg in no_churn) and any("tenure" in msg for msg in text_tenure))

check('🚀 C5', _test, hint='telco_clean must give [] (TotalCharges may be missing: nullable). A missing column is a message, not a KeyError; every message names its column.')

**✍️ Your answer** — Where in a project would you call `validate()`, and what should happen when it returns problems?

> *(double-click this cell and write your answer here)*

## C6 · Official statistics from Eurostat 🌶️ (15 min) · 📈

**Why it matters.** Statistical offices publish their data through APIs. If you read a table straight from the URL, your analysis updates itself when the next month is published, with no manual download from a web page.

The next cell holds a query for Eurostat's **HICP** (harmonised index of consumer prices) in **SDMX-CSV**, a standard format of official statistics. The URL itself is the query: dataset `prc_hicp_minr`, then the key `M.RCH_A.TOTAL.SI+EA+EU` = monthly · annual rate of change (that is, inflation in %) · all items · Slovenia, the euro area and the EU, from January 2015.

**Build**
1. `hicp_long`: read the URL with `pd.read_csv`, keep `geo`, `TIME_PERIOD` (renamed to `date`, as a datetime) and `OBS_VALUE` (renamed to `inflation`): one row per country and month;
2. `hicp_wide`: one row per month and one column per country (`SI`, `EA`, `EU`), and a plot with a title, axis labels and the ECB's 2 % target as a dashed line;
3. `gap_si_ea`: the average of SI − EA in percentage points, and `months_above_5`: the number of months above 5 % for each country.

Then interpret the result in 2–3 sentences in the answer cell below.

**Hints:** `.rename(columns={...})` · `pd.to_datetime()` · `.pivot(index="date", columns="geo", values="inflation")` · `ax.axhline(2, linestyle="--")` · `(hicp_wide > 5).sum()`.

In [ ]:
# The URL is the query: dataset prc_hicp_minr, key = frequency.unit.coicop18.geo, data from January 2015
HICP_URL = ("https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/prc_hicp_minr/"
            "M.RCH_A.TOTAL.SI+EA+EU?format=SDMX-CSV&startPeriod=2015-01")

In [ ]:
# 🚀 Challenge: Read Eurostat's HICP inflation for SI, EA and EU from HICP_URL into the tidy table hicp_long
#    (geo, date, inflation), pivot it to hicp_wide (one column per country), plot it,
#    and compute gap_si_ea (average SI − EA) and months_above_5 (months above 5 % per country).
# hicp_raw = pd.read_csv(HICP_URL)
# hicp_long = hicp_raw[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={...})
# hicp_long["date"] = ...
# hicp_wide = hicp_long.pivot(...)
# ax = hicp_wide.plot(...)
# gap_si_ea = ...
# months_above_5 = ...


In [ ]:
#@title ✅ Check — 🚀 C6 { display-mode: "form" }
def _test():
    wide = hicp_wide
    expected_gap = (wide["SI"] - wide["EA"]).mean()             # recomputed from your table: no fixed latest month
    expected_above = (wide[["SI", "EA", "EU"]] > 5).sum()
    above = pd.Series(months_above_5)
    return (isinstance(wide.index, pd.DatetimeIndex) and wide.index.min() == pd.Timestamp("2015-01-01")
            and wide.index.is_unique and wide.index.is_monotonic_increasing and len(wide) >= 120
            and {"SI", "EA", "EU"} <= set(wide.columns)
            and {"geo", "date", "inflation"} <= set(hicp_long.columns) and set(hicp_long["geo"]) == {"SI", "EA", "EU"}
            and wide.loc["2022", "SI"].max() > 9                       # a fact from history: the 2022 inflation peak
            and abs(float(gap_si_ea) - expected_gap) < 0.01
            and all(int(above[g]) == int(expected_above[g]) for g in ["SI", "EA", "EU"]))

check('🚀 C6', _test, hint='hicp_wide needs a DatetimeIndex from January 2015 and the columns SI, EA, EU; gap_si_ea = mean of SI − EA; months_above_5 = (hicp_wide > 5).sum().')

**✍️ Your answer** — Interpret the result in 2–3 sentences: how does Slovenia's inflation compare with the euro area's, and what does the average gap hide?

> *(double-click this cell and write your answer here)*

## 🏁 Open challenge · Several cities in one request (20 min) · 📈💻

The Open-Meteo archive accepts **comma-separated lists** of coordinates (`latitude=45.55,46.05&longitude=13.73,14.51`) and then returns a **list** of JSON objects: one per location, in the same order. One request instead of a loop is faster and kinder to a free API.

Build `weather_long`: a **tidy long table** with one row per city and day (columns `city`, `date`, `temperature_2m_max`, `temperature_2m_min`, `precipitation_sum`) for at least three cities in 2025. Then plot the **monthly mean of the daily maximum temperature**, one line per city, with a title and axis labels. Which city is warmest in summer? Which has the largest range between its hottest day and its coldest night?

**Hints:** `",".join(...)` · `pd.concat([... .assign(city=name) for ...], ignore_index=True)` · `pivot_table(index=..., columns="city", values=..., aggfunc="mean")` · `.plot(title=..., xlabel=..., ylabel=...)`.

In [ ]:
# 🚀 🏁 Open challenge: Fetch the 2025 daily weather for several cities in ONE Open-Meteo request, build the tidy
#    long table weather_long (city, date, ...) and plot the monthly mean maximum temperature per city.
# cities = {"Koper": (45.55, 13.73), "Ljubljana": (46.05, 14.51), ...}
# params_multi = {**params, "latitude": ",".join(...), "longitude": ",".join(...)}
# locations = requests.get(METEO_URL, params=params_multi, timeout=60).json()
# weather_long = pd.concat([...], ignore_index=True)
# ...


**✍️ Your answer** — Which city is warmest in summer, which has the largest temperature range, and why is the long format handy?

> *(double-click this cell and write your answer here)*

In [ ]:
progress()

---
## Going further (optional, self-study)

### Local setup in 10 minutes
1. **Install uv.** Windows (PowerShell): `powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"` · macOS/Linux: `curl -LsSf https://astral.sh/uv/install.sh | sh`
2. **Create a project with its own environment:** `uv init dm-project` → `cd dm-project` → `uv add pandas scikit-learn matplotlib seaborn xgboost jupyter`
3. **Install VS Code** with the *Python* and *Jupyter* extensions, open the folder, open a notebook and select the `.venv` kernel.
4. **Alternative — Miniforge (conda):** `conda create -n dm python=3.13 pandas scikit-learn jupyter` → `conda activate dm`

### More ways to get data in
- Google Drive in Colab: `from google.colab import drive; drive.mount("/content/drive")`
- Very large CSV files in pieces: `pd.read_csv(url, chunksize=100_000)`
- Every sheet of a workbook at once: `pd.read_excel(path, sheet_name=None)`
- **polars** and **DuckDB**, fast alternatives for big tables (both preinstalled in Colab): see Part C, challenges C3 and C4

### References
[pandas IO tools](https://pandas.pydata.org/docs/user_guide/io.html) · [requests](https://requests.readthedocs.io/) · [Open-Meteo API docs](https://open-meteo.com/en/docs) · [uv](https://docs.astral.sh/uv/) · [DuckDB](https://duckdb.org/docs/) · [polars](https://docs.pola.rs/)

**Data sources and licences:** Wine Quality — Cortez et al. (2009), UCI, CC BY 4.0 · Energy Efficiency — Tsanas & Xifara (2012), UCI, CC BY 4.0 · Weather — Open-Meteo.com, CC BY 4.0 · Telco Customer Churn — IBM sample data · Inflation (Part C) — Source: Eurostat, HICP – ECOICOP ver.2 – indices and rates of change, monthly data (`prc_hicp_minr`), [doi:10.2908/PRC_HICP_MINR](https://doi.org/10.2908/PRC_HICP_MINR); reuse authorised provided the source is acknowledged (Commission Decision 2011/833/EU).